In [13]:
import numpy as np
import pymc as pm
import arviz as az
import xarray as xr

In [2]:
np.random.seed(1)

#### Competing Bayesian Regression Models

In [3]:
# Simulating data.

n = 300

x1 = np.random.normal(0, 1, n)
x2 = np.random.normal(0, 1, n)

true_intercept = -0.5
true_beta1 = 2.0
true_beta2 = 1.0

linear = (
    true_intercept
    + true_beta1 * x1
    + true_beta2 * x2
)

p = 1 / (1 + np.exp(-linear))

y = np.random.binomial(1, p)

print("Sample size:", n)

Sample size: 300


In [4]:
# Model 1: Simple logistic regression.

with pm.Model() as simple_model:

    intercept = pm.Normal(
        "intercept",
        mu=0,
        sigma=5
    )

    beta1 = pm.Normal(
        "beta1",
        mu=0,
        sigma=5
    )

    probability = pm.math.sigmoid(
        intercept + beta1 * x1
    )

    outcome = pm.Bernoulli(
        "outcome",
        p=probability,
        observed=y
    )

    simple_trace = pm.sample(
        draws=2000,
        tune=1000,
        random_seed=1,
        return_inferencedata=True
    )

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [intercept, beta1]


Output()

Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 4 seconds.


In [5]:
# Model 2: More complex logistic regression.

with pm.Model() as complex_model:

    intercept = pm.Normal(
        "intercept",
        mu=0,
        sigma=5
    )

    beta1 = pm.Normal(
        "beta1",
        mu=0,
        sigma=5
    )

    beta2 = pm.Normal(
        "beta2",
        mu=0,
        sigma=5
    )

    probability = pm.math.sigmoid(
        intercept
        + beta1 * x1
        + beta2 * x2
    )

    outcome = pm.Bernoulli(
        "outcome",
        p=probability,
        observed=y
    )

    complex_trace = pm.sample(
        draws=2000,
        tune=1000,
        random_seed=1,
        return_inferencedata=True
    )

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [intercept, beta1, beta2]


Output()

Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 4 seconds.


#### Compare Models Using LOO and WAIC

In [6]:
with simple_model:
    simple_trace = pm.compute_log_likelihood(simple_trace)

with complex_model:
    complex_trace = pm.compute_log_likelihood(complex_trace)

Output()

Output()

In [7]:
# Computing LOO.

simple_loo = az.loo(simple_trace)

complex_loo = az.loo(complex_trace)

print("Simple model:\n", simple_loo)

print("\nComplex model:\n", complex_loo)

Simple model:
 Computed from 8000 posterior samples and 300 observations log-likelihood matrix.

         Estimate       SE
elpd_loo  -156.05     8.55
p_loo        1.98        -
------

Pareto k diagnostic values:
                         Count   Pct.
(-Inf, 0.70]   (good)      300  100.0%
   (0.70, 1]   (bad)         0    0.0%
    (1, Inf)   (very bad)    0    0.0%


Complex model:
 Computed from 8000 posterior samples and 300 observations log-likelihood matrix.

         Estimate       SE
elpd_loo  -128.72     9.53
p_loo        2.97        -
------

Pareto k diagnostic values:
                         Count   Pct.
(-Inf, 0.70]   (good)      300  100.0%
   (0.70, 1]   (bad)         0    0.0%
    (1, Inf)   (very bad)    0    0.0%



In [14]:
# Computing WAIC.

def compute_waic(trace):
    log_likelihood = trace.log_likelihood["outcome"]

    # Combine chains and draws into one posterior-sample dimension
    log_likelihood = log_likelihood.stack(sample=("chain", "draw"))

    # Convert to NumPy: (samples, observations)
    log_lik = log_likelihood.values.T

    # lppd: log pointwise predictive density
    max_log_lik = np.max(log_lik, axis=0)
    lppd_i = (
        max_log_lik
        + np.log(np.mean(np.exp(log_lik - max_log_lik), axis=0))
    )

    # Effective number of parameters
    p_waic_i = np.var(log_lik, axis=0, ddof=1)

    # WAIC
    elpd_waic = np.sum(lppd_i - p_waic_i)
    waic = -2 * elpd_waic

    return {
        "elpd_waic": elpd_waic,
        "p_waic": np.sum(p_waic_i),
        "waic": waic,
    }


simple_waic = compute_waic(simple_trace)
complex_waic = compute_waic(complex_trace)

print("Simple model:")
print(simple_waic)

print("\nComplex model:")
print(complex_waic)

Simple model:
{'elpd_waic': np.float64(-156.04900119145202), 'p_waic': np.float64(1.9821600596645284), 'waic': np.float64(312.09800238290404)}

Complex model:
{'elpd_waic': np.float64(-128.71741819836825), 'p_waic': np.float64(2.962574254072942), 'waic': np.float64(257.4348363967365)}


In [17]:
# Comparing models.

comparison = az.compare(
    {"Simple Model": simple_trace, "Complex Model": complex_trace},
    method="stacking"
)

print(comparison)

               rank  elpd_diff  dse  p_worse diag_diff diag_elpd    p   elpd  \
Complex Model     0        0.0  0.0      NaN                      3.0 -130.0   
Simple Model      1      -30.0  7.3      1.0                      2.0 -160.0   

                se  weight  
Complex Model  9.5    0.96  
Simple Model   8.5    0.04  


In [18]:
# Interpreting the results.

best_model = comparison.index[0]

print("Preferred Model: ", best_model)

if best_model == "Simple Model":
    print("The simpler model has better expected predictive performance.")
else:
    print("The more complex model has better expected predictive performance.")

Preferred Model:  Complex Model
The more complex model has better expected predictive performance.
